In [0]:
install.packages("ranger") 

In [0]:
%r
# 1. Carregar as bibliotecas necessárias
library(sparklyr)
library(dplyr)
library(caret) # Ou tidymodels / ranger para Random Forest

# Conectar ao Spark 
sc <- spark_connect(method = "databricks")

# 2. Carregar os dados da tabela Gold do Data Lake
df_gold <- tbl(sc, "anvisa.g_vigimed_features_ml")

# 3.
# Se  demorar muito (se a tabela for gigante), o ideal é usar o sparklyr + ml_random_forest. 
df_R <- df_gold %>% 
  mutate(TARGET_IS_GRAVE = as.numeric(TARGET_IS_GRAVE)) %>%
  collect()

# Converter colunas categóricas para factor
categorical_cols <- c("SEXO", "UF", "TIPO_NOTIFICACAO", "RECEBIDO_DE")
df_R[categorical_cols] <- lapply(df_R[categorical_cols], as.factor)

# Garantir que o target seja fator para classificação binária
df_R$TARGET_IS_GRAVE <- as.factor(df_R$TARGET_IS_GRAVE)

# 4. Divisão em Treino (80%) e Teste (20%) com semente fixa
set.seed(42)
train_index <- createDataPartition(df_R$TARGET_IS_GRAVE, p = 0.8, list = FALSE)
train_df <- df_R[train_index, ]
test_df  <- df_R[-train_index, ]

print(paste("Registros de Treino:", nrow(train_df)))
print(paste("Registros de Teste:", nrow(test_df)))

# 5. Treinamento do Modelo Random Forest usando o pacote 'ranger' (+rápido) posso usar o 'caret' aqui tbm
library(ranger)

# Ajustando o modelo Random Forest
modelo_rf <- ranger(
  formula = TARGET_IS_GRAVE ~ ., 
  data = train_df,
  num.trees = 100,
  max.depth = 8,
  seed = 42,
  probability = TRUE # Permite gerar probabilidades para cálculo de AUC
)

print(modelo_rf)

# 6. Geração de Previsões na Base de Teste
predicoes <- predict(modelo_rf, data = test_df)
# Extraindo as classes preditas (ajustando o limiar de corte para 0.5)
probabilidades <- predicoes$predictions[, 2] # Probabilidade da classe 1 (Grave)
classe_predita <- ifelse(probabilidades > 0.5, 1, 0)

# 7. Avaliação e Matriz de Confusão
test_df$predicao_fator <- as.factor(classe_predita)
matriz_confusao <- confusionMatrix(test_df$predicao_fator, test_df$TARGET_IS_GRAVE, positive = "1")

print(matriz_confusao)

In [0]:
    print(matriz_confusao)